# FNO · 히터 위치에 따른 알루미늄 판의 온도

가로·세로 64 mm, 두께 1 mm인 알루미늄 판에 8×8 mm 히터 하나를 붙입니다. 네 가장자리는 단열이고 넓은 표면에서는 25°C 공기로 열이 빠져나갑니다. 같은 2 W 히터를 중앙에 놓을 때와 모서리 가까이 놓을 때, 어느 쪽에서 최고 온도가 더 높을까요?

한 사례는 64×64 격자에 그린 지도 두 장입니다.

- **입력 `f`**는 히터가 단위 면적당 공급하는 전력 $q$(W/m²)입니다. 2 W 히터라면 히터가 덮은 64칸에는 각각 31,250 W/m², 나머지 칸에는 0이 들어갑니다.
- **학습 목표 `u`**는 정상상태 온도 $T$(°C)입니다. 히터를 켜 두어 판의 온도가 더 이상 변하지 않을 때의 값을 열전도 방정식으로 계산합니다.

$$-kt\,\nabla^2 T + h(T-T_{\mathrm{air}})=q.$$

$k=200$ W/(m·K)는 알루미늄의 열전도율, $t=0.001$ m는 판의 두께, $h=20$ W/(m²·K)는 표면의 유효 열전달계수입니다. 주변 공기 온도 $T_{\mathrm{air}}$는 25°C입니다.

FNO는 여러 위치에서 전력이 1, 2, 3, 4 W인 히터의 사례를 보고 두 지도 사이의 관계를 학습합니다. 시험에는 학습에 쓰지 않은 위치의 히터만 등장합니다.


## 1. 히터 위치와 온도 분포

아래 두 셀을 실행하세요. 첫 번째 학습 사례의 발열 밀도와 온도를 나란히 그립니다.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/pinn_basics.py").is_file()
             and (p / "02_FNO.ipynb").is_file()), None)
if ROOT is None:
    raise RuntimeError("워크숍 교재 폴더 안에서 이 노트북을 여세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("교재:", ROOT)
print("Python:", sys.executable)


In [ ]:
import json
from uuid import uuid4
import numpy as np
import torch
from torch.utils.data import TensorDataset
from physicsnemo.models.fno import FNO
from src.operator_training import run
from IPython.display import Image, display

if not torch.cuda.is_available():
    raise RuntimeError("00_Start_Here의 GPU 확인부터 진행하세요.")
DEVICE = "cuda"
DATA_DIR = ROOT / "data/heated_plate"
CONFIG = ROOT / "config/config_FNO.yaml"
if not all((DATA_DIR / f"{split}.hdf5").exists() for split in ("train", "val", "test")):
    raise RuntimeError("00_Start_Here에서 데이터를 준비하세요.")
FNO_COMPLETED = False


import h5py
from io import BytesIO
from matplotlib import pyplot as plt

def show_plate_figure(figure):
    with BytesIO() as png:
        figure.savefig(png, format="png", dpi=110)
        display(Image(data=png.getvalue()))
    plt.close(figure)

with h5py.File(DATA_DIR / "train.hdf5", "r") as data:
    q = data["f"][0, 0]
    temperature = data["u"][0, 0]
    power = float(data["power_w"][0])

print(f"학습 사례 하나: 히터 {power:g} W, 가열되는 칸 {np.count_nonzero(q)}개")
print(f"발열량 {q.min():,.0f}~{q.max():,.0f} W/m², 판 온도 {temperature.min():.2f}~{temperature.max():.2f} °C")
fig, axes = plt.subplots(1, 2, figsize=(8, 3.2), constrained_layout=True)
for ax, field, title, unit, cmap in zip(axes, (q, temperature),
        ("Heater power density", "Reference plate temperature"), ("W/m²", "°C"), ("magma", "inferno")):
    image = ax.imshow(field.T, origin="lower", extent=(0, 64, 0, 64), cmap=cmap)
    ax.set(title=title, xlabel="x (mm)", ylabel="y (mm)")
    fig.colorbar(image, ax=ax, label=unit)
show_plate_figure(fig)


## 2. 데이터셋과 FNO 만들기

`build_datasets`는 학습·검증·시험용 텐서를 PyTorch 데이터셋으로 감쌉니다. `build_model`은 발열 밀도 지도 한 장을 받아 온도 지도 한 장을 예측하는 2차원 FNO를 만듭니다. 정규화와 학습은 뒤의 셀에서 처리합니다.

AI 코딩 도구로 구현할 때는 아래 요청문과 함수 틀을 함께 붙여 넣으세요. 받은 두 함수로 다음 셀을 바꿔 실행합니다. [예제 코드](examples/fno.py)로 진행할 때는 셀을 그대로 실행하세요.

> 아래 두 함수를 완성하세요. `TensorDataset`과 `FNO`는 이미 불러왔습니다. 첫 함수에서는 각 입력 쌍을 `TensorDataset(*pairs)`로 감싸 학습·검증·시험 순서대로 반환하세요. 두 번째 함수는 `FNO(in_channels=1, out_channels=1, dimension=2, **model_config)`를 반환합니다. 두 함수의 이름과 인자는 그대로 둡니다.

```python
def build_datasets(train_pairs, val_pairs, test_pairs):
    pass

def build_model(model_config, grid_size):
    pass
```


In [ ]:
from examples.fno import build_datasets, build_model


<a id="fno-split-check"></a>

## 3. 검증·시험 데이터의 순서 확인

검증과 시험 데이터는 각각 1,000개이고 격자 크기도 같습니다. 두 묶음이 뒤바뀌어도 크기만 보면 알아낼 수 없습니다.

아래 셀은 각 파일에서 첫 번째 사례를 읽고, 방금 만든 함수가 반환한 입력값과 정답을 파일의 값과 대조합니다. 일부러 검증·시험 순서를 바꾼 두 번째 호출에서 오류를 잡아내는지 확인하세요.


In [ ]:
import h5py
from src.operator_training import checked_datasets

sample_pairs = []
for split in ("train", "val", "test"):
    with h5py.File(DATA_DIR / f"{split}.hdf5", "r") as data:
        sample_pairs.append((torch.from_numpy(data["f"][:1]), torch.from_numpy(data["u"][:1])))

checked_datasets(sample_pairs, build_datasets)
print("원래 순서로 검사 통과. 검증·시험 입력의 크기:", sample_pairs[1][0].shape, sample_pairs[2][0].shape)


def swap_val_and_test(train, val, test):
    return build_datasets(train, test, val)


try:
    checked_datasets(sample_pairs, swap_val_and_test)
except ValueError as error:
    if not str(error).startswith("val.f:"):
        raise
    print("검증·시험 교환을 감지했습니다. 검증 입력 f가 원래 값과 다릅니다.")
else:
    raise AssertionError("검증·시험을 바꾼 결과가 검사를 통과했습니다.")


### 두 번만 학습해 보기

같은 검사로 전체 데이터를 확인한 뒤 모델을 두 번 업데이트합니다. 이 짧은 실행이 끝나면 다음 절에서 새 모델을 학습하세요.


In [ ]:
SMOKE_RUN = ROOT / "runs" / f"fno-check-{uuid4().hex[:10]}"
smoke_metrics = run(1, build_model, dataset_builder=build_datasets, argv=[
    "--config", str(CONFIG), "--data-dir", str(DATA_DIR), "--device", DEVICE,
    "--steps", "2", "--seed", "42", "--output-dir", str(SMOKE_RUN)])
print("2회 업데이트를 마쳤습니다.")


## 4. FNO 학습

히터의 총전력과 표면에서 빠져나가는 열이 균형을 이루므로 판의 평균 온도는 미리 계산할 수 있습니다. FNO는 각 위치가 이 평균보다 얼마나 뜨겁거나 차가운지를 학습합니다. 예측할 때는 평균 온도를 다시 더합니다.

배치 크기는 32이고 Adam으로 3,000번 업데이트합니다. 학습이 끝나면 결과 폴더의 경로가 표시됩니다.


In [ ]:
FNO_COMPLETED = False
FNO_RUN = ROOT / "runs" / f"fno-{uuid4().hex[:10]}"
fno_metrics = run(1, build_model, dataset_builder=build_datasets, argv=[
    "--config", str(CONFIG), "--data-dir", str(DATA_DIR), "--device", DEVICE,
    "--steps", "3000", "--seed", "42", "--output-dir", str(FNO_RUN)])
FNO_COMPLETED = True
print("결과:", FNO_RUN.relative_to(ROOT))


## 5. 새로운 히터 위치에서 예측 오차 확인

시험 사례 1,000개의 결과를 읽습니다. `Test temperature MAE`는 모든 격자점의 평균 절대 온도 오차입니다. `Test peak temperature MAE`는 각 판의 최고 온도를 몇 도나 틀렸는지 평균한 값입니다. 상대 L2 오차는 25°C에서의 온도 상승을 기준으로 계산합니다. 열수지 잔차는 각 칸에 열전도 방정식을 적용했을 때 남는 차이로, 단위는 W/m²입니다.

아래 그림은 시험 사례 하나의 수치해, FNO 예측, 두 값의 차이(예측 − 수치해)를 보여 줍니다. 오른쪽 그림에서 오차가 가장 큰 곳을 찾아보세요.


In [ ]:
if not FNO_COMPLETED:
    raise RuntimeError("4절에서 모델을 3,000회 학습한 뒤 실행하세요.")
saved_metrics = json.loads((FNO_RUN / "metrics.json").read_text())
test = saved_metrics["test"]
print(f"Test temperature MAE: {test['mae_c']:.3f} °C")
print(f"Test peak temperature MAE: {test['peak_mae_c']:.3f} °C")
print(f"Test temperature RMSE: {test['rmse']:.3f} °C")
print(f"Test relative L2 (temperature rise): {100 * test['relative_l2']:.2f}%")
print(f"Test heat balance residual RMSE: {test['pde_rmse_w_m2']:.1f} W/m²")
display(Image(filename=str(FNO_RUN / "preview.png")))


### 같은 2 W 히터를 중앙에서 모서리 근처로 옮기면?

첫째 줄에는 히터를 중앙 (32, 32) mm에 놓았을 때, 둘째 줄에는 모서리 근처 (12, 12) mm에 놓았을 때의 결과가 나옵니다. 수치해의 최고 온도는 어느 쪽이 더 높은가요? 오른쪽 열의 FNO가 이 차이를 얼마나 잘 예측했는지 비교해 보세요.


In [ ]:
from examples.plate_prediction import plot_heater_cases

with np.load(FNO_RUN / "predictions.npz", allow_pickle=False) as saved:
    heat_flux = saved["f"][:2, 0]
    targets = saved["u_true"][:2, 0]
    predictions = saved["u_pred"][:2, 0]

labels = ("Centre (32, 32) mm", "Near corner (12, 12) mm")
for location, reference, predicted in zip(labels, targets, predictions):
    print(f"{location}: reference peak {reference.max():.2f} °C, "
          f"FNO peak {predicted.max():.2f} °C, reference mean {reference.mean():.2f} °C")
show_plate_figure(plot_heater_cases(list(zip(heat_flux, targets, predictions)), labels))


### 선택 실습 · 판의 평균 온도

판 가장자리로는 열이 빠져나가지 않으므로, 정상상태에서는 표면을 통해 나가는 열이 히터의 전력 $P$와 같습니다. 면적이 $A$인 판의 평균 온도는 $T_{\mathrm{air}}+P/(hA)$입니다. 아래 셀에서 두 위치의 평균 온도와 최고 온도를 각각 비교하세요.


In [ ]:
plate_area = 0.064 ** 2
mean_from_energy = 25 + 2 / (20 * plate_area)
print(f"열수지에서 구한 평균 온도: {mean_from_energy:.2f} °C")
for position, temperature in zip(("중앙", "모서리 근처"), targets):
    print(f"{position}: 평균 {temperature.mean():.2f} °C, 최고 {temperature.max():.2f} °C")


### 선택 실습 · 히터의 위치와 전력 바꾸기

아래 셀의 첫 두 줄에서 히터 중심 좌표(mm)와 전력(W)을 바꿀 수 있습니다. 두 좌표는 각각 4~60 사이의 정수이고, 전력은 1, 2, 3, 4 W 중 하나입니다.

먼저 2 W에서 중심을 `(32, 32)`와 `(12, 12)`로 바꿔 앞의 두 사례를 재현하세요. 다음에는 위치를 고정하고 전력을 4 W로 높여 보세요. 25°C에서의 온도 상승은 어떻게 달라지나요?

이 셀은 4절에서 저장한 모델로 예측하고 같은 조건의 수치해를 새로 계산합니다. 임의로 고른 위치는 학습에 사용한 적이 있을 수 있습니다.


In [ ]:
heater_center_mm = (32, 32)
power_w = 2.0

from examples.plate_prediction import plot_heater_cases, predict_heater_case
if not FNO_COMPLETED:
    raise RuntimeError("4절에서 모델을 3,000회 학습한 뒤 실행하세요.")
design = predict_heater_case(FNO_RUN / "model.pt", heater_center_mm, power_w, build_model, DEVICE)
q, reference, predicted = design
print(f"히터 {power_w:g} W, 중심 {heater_center_mm} mm")
print(f"수치해 최고 온도 {reference.max():.2f} °C, FNO 최고 온도 {predicted.max():.2f} °C")
label = f"{power_w:g} W, {heater_center_mm} mm"
show_plate_figure(plot_heater_cases([design], [label]))


## 6. PINN과 FNO 비교

| | 줄의 처짐을 구한 PINN (Lab 1.1) | 판의 온도를 구한 FNO |
| --- | --- | --- |
| 입력 | 줄 위의 위치 $x$ | 판 전체의 발열 밀도 지도 $q$ (W/m²) |
| 출력 | 그 위치의 수직 변위 $u(x)$ | 판 전체의 정상상태 온도 $T$ (°C) |
| 학습에 쓴 정보 | 줄의 방정식과 양끝의 높이 | 히터의 위치·전력을 바꿔 계산한 발열 밀도와 온도 지도 쌍 |
| 예측의 비교 기준 | 해석해 | 학습에서 제외한 히터 위치의 수치해 |

같은 방식으로 다른 물리량의 분포도 예측할 수 있습니다. [Darcy 유동 예제](https://docs.nvidia.com/physicsnemo/latest/user-guide/simple_training_example.html#using-built-in-models)에서는 물이 지층을 얼마나 잘 통과하는지 나타내는 투과율 지도로부터 압력 분포를 구합니다.

Brev에서 실습했다면 노트북을 저장하고 콘솔에서 내 인스턴스의 `Stop`을 누르세요. 상태가 `Stopped`로 바뀌면 종료된 것입니다.


[이전 실습 · PINN으로 줄의 처짐 구하기](01_PINN.ipynb) · [실습 목차](00_Start_Here.ipynb)
